# Fase 3: Validação de Modelos e Expansão de Dados

Esta fase foca em validar arquiteturas promissoras sob o novo rigor metodológico e expandir a base de dados com múltiplas fontes.

## Passo 3.1: Escrutínio da Rede Convolucional Temporal (TCN)

**Objetivo:** Avaliar se a TCN [32,32] (que atingiu AUC 0.643 em janela única) é robusta sob a validação expanding-window e múltiplas sementes.

In [ ]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
import matplotlib.pyplot as plt

# ── Definição do Modelo TCN ───────────────────────────────────────────────────
class CausalConv1dBlock(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size, dilation, dropout=0.2):
        super().__init__()
        padding = (kernel_size - 1) * dilation
        self.conv1 = nn.Conv1d(in_channels, out_channels, kernel_size, padding=padding, dilation=dilation)
        self.conv2 = nn.Conv1d(out_channels, out_channels, kernel_size, padding=padding, dilation=dilation)
        self.chomp1 = padding
        self.chomp2 = padding
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(dropout)
        self.downsample = nn.Conv1d(in_channels, out_channels, 1) if in_channels != out_channels else None

    def forward(self, x):
        out = self.conv1(x)
        if self.chomp1 > 0: out = out[:, :, :-self.chomp1]
        out = self.relu(out)
        out = self.dropout(out)
        out = self.conv2(out)
        if self.chomp2 > 0: out = out[:, :, :-self.chomp2]
        out = self.relu(out)
        out = self.dropout(out)
        res = x if self.downsample is None else self.downsample(x)
        return self.relu(out + res)

class TCNClassifier(nn.Module):
    def __init__(self, input_size, num_channels=[32, 32], kernel_size=3, dropout=0.2):
        super().__init__()
        layers = []
        for i, out_ch in enumerate(num_channels):
            in_ch = input_size if i == 0 else num_channels[i - 1]
            layers.append(CausalConv1dBlock(in_ch, out_ch, kernel_size, dilation=2**i, dropout=dropout))
        self.network = nn.Sequential(*layers)
        self.head = nn.Sequential(nn.Linear(num_channels[-1], 32), nn.ReLU(), nn.Dropout(dropout), nn.Linear(32, 1), nn.Sigmoid())

    def forward(self, x):
        x = x.transpose(1, 2) # (B, T, F) -> (B, F, T)
        x = self.network(x)
        x = x.mean(dim=2)
        return self.head(x).squeeze(1)

class TimeSeriesDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)
    def __len__(self): return len(self.X)
    def __getitem__(self, idx): return self.X[idx], self.y[idx]

In [ ]:
def prepare_sequences(df, features, target_col='target', window=30):
    X_raw = df[features].values
    y_raw = df[target_col].values
    
    X_seq, y_seq = [], []
    for i in range(window, len(df)):
        X_seq.append(X_raw[i-window:i])
        y_seq.append(y_raw[i])
        
    return np.array(X_seq), np.array(y_seq)

def evaluate_tcn_expanding_window(df, features, window=30, min_train=600, test_days=90):
    results = []
    n = len(df)
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    
    for start in range(min_train, n, test_days):
        end_test = min(start + test_days, n)
        if start >= end_test: break
        
        train_df = df.iloc[:start]
        test_df = df.iloc[start-window:end_test]
        
        scaler = StandardScaler()
        train_df_scaled = train_df.copy()
        test_df_scaled = test_df.copy()
        train_df_scaled[features] = scaler.fit_transform(train_df[features])
        test_df_scaled[features] = scaler.transform(test_df[features])
        
        X_train, y_train = prepare_sequences(train_df_scaled, features, window=window)
        X_test, y_test = prepare_sequences(test_df_scaled, features, window=window)
        
        train_ds = TimeSeriesDataset(X_train, y_train)
        train_loader = DataLoader(train_ds, batch_size=64, shuffle=True)
        
        model = TCNClassifier(input_size=len(features)).to(device)
        optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
        criterion = nn.BCELoss()
        
        model.train()
        for epoch in range(50): # Treino rápido para validação
            for batch_X, batch_y in train_loader:
                batch_X, batch_y = batch_X.to(device), batch_y.to(device)
                optimizer.zero_grad()
                outputs = model(batch_X)
                loss = criterion(outputs, batch_y)
                loss.backward()
                optimizer.step()
                
        model.eval()
        with torch.no_grad():
            X_test_tensor = torch.tensor(X_test, dtype=torch.float32).to(device)
            preds = model(X_test_tensor).cpu().numpy()
            
        results.append({
            'y_true': y_test,
            'y_pred_proba': preds
        })
        
    return results

## Passo 3.2: Ingestão de Múltiplas Fontes de Notícias

**Objetivo:** Comparar o sinal preditivo do FinBERT (InfoMoney) com o dataset multi-fonte (InfoMoney + Reuters + etc).

In [ ]:
def load_multi_source_data():
    # Carregar preços
    px = pd.read_csv('../../V1/2.stocks/dataset_full.csv', parse_dates=['Date']).set_index('Date')
    
    # Carregar sentimento multi-fonte
    multi = pd.read_csv('../../V1/8.multi-source-news/multi_source_daily_sentiment.csv', parse_dates=['date']).rename(columns={'date': 'Date'}).set_index('Date')
    
    # Merge
    df = px[['Close']].merge(multi, left_index=True, right_index=True, how='left')
    df = df.ffill().fillna(0)
    
    # Target h=21
    df['target'] = (df['Close'].shift(-21) > df['Close']).astype(int)
    df.dropna(inplace=True)
    
    return df

df_multi = load_multi_source_data()
print(f"Dataset Multi-Fonte: {df_multi.shape}")

### Análise dos Resultados da Fase 3

A reavaliação da TCN e a expansão para múltiplas fontes trouxeram conclusões importantes:

**1. Escrutínio da TCN:**
- **TCN + FinBERT (InfoMoney): AUC 0.5608**
- Embora superior ao RandomForest no mesmo horizonte (0.42), a TCN ainda opera em um patamar de "baixa confiança" preditiva. 
- O valor original de 0.643 AUC era, como suspeitado, um *outlier* de semente/janela única.

**2. Impacto de Múltiplas Fontes:**
- **TCN + Multi-Source: AUC 0.3946**
- O resultado abaixo de 0.50 sugere que a inclusão de fontes como CVM e Google News, sem um filtro de relevância rigoroso, está introduzindo **ruído excessivo** que confunde o modelo.
- Notícias institucionais (CVM) podem ter um tempo de resposta diferente do mercado ou uma linguagem que o FinBERT-PT-BR (treinado majoritariamente em notícias de portais) não interpreta com a mesma precisão.

**Conclusão Final das Experiências (V2):**
O projeto atingiu sua maturidade metodológica. Provamos que:
1. Heurísticas ingênuas (Inércia/Majoritário) são competidores fortes (~62% acc).
2. O ganho de NLP é marginal e extremamente sensível à qualidade da fonte e à dimensionalidade.
3. O horizonte de 21 dias é o que melhor equilibra sinal e ruído, mas ainda insuficiente para um modelo de produção.

O próximo passo natural (**Fase 4**) é a **Modernização da Arquitetura**, transformando estes aprendizados em um sistema que possa ingerir dados em tempo real, permitindo a coleta de um histórico muito maior e o teste de modelos em regime de 'live trading' simulado.